# Week 5 Lab: Optimization for Training Deep Models

**Course:** Deep Learning (COE 443) — Istinye University

**Instructor:** Asst. Prof. Dr. Yigit Bekir Kaya

**Reference:** Goodfellow, Bengio, Courville — *Deep Learning*, Chapter 8

---

## Objectives

In this lab you will:

1. **Implement vanilla SGD from scratch** and visualize its trajectory on 2D loss surfaces
2. **Explore optimization challenges** — ill-conditioning, saddle points, and ravines
3. **Build SGD with momentum and Nesterov momentum** from scratch and compare their paths
4. **Implement adaptive optimizers from scratch** — AdaGrad, RMSProp, and Adam
5. **Compare parameter initialization strategies** — random, Xavier, He initialization
6. **Experiment with learning rate schedules** — step decay, cosine annealing, warmup
7. **Run a grand optimizer shootout** on Fashion-MNIST with a real neural network

Each part connects directly to the lecture slides and textbook equations.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from copy import deepcopy

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Load Fashion-MNIST

We reuse Fashion-MNIST from last week. This time we focus on how different **optimizers** and **learning rate schedules** affect convergence speed and final accuracy.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Image shape:      {train_dataset[0][0].shape}")

## Helper: Training Loop for PyTorch Models

In [ ]:
class MLP(nn.Module):
    """Standard MLP for Fashion-MNIST experiments."""
    def __init__(self, init_method='default'):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        if init_method != 'default':
            self._init_weights(init_method)

    def _init_weights(self, method):
        for m in [self.fc1, self.fc2, self.fc3]:
            if method == 'xavier':
                nn.init.xavier_uniform_(m.weight)
            elif method == 'he':
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            elif method == 'small_random':
                nn.init.normal_(m.weight, 0, 0.01)
            elif method == 'large_random':
                nn.init.normal_(m.weight, 0, 1.0)
            elif method == 'zeros':
                nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        x = x.view(-1, 784)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


def train_model(model, train_loader, test_loader, optimizer, scheduler=None,
                epochs=30, verbose=True):
    """General training loop. Returns per-epoch metrics."""
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'lr': []}

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784)
            out = model(X_batch)
            loss = criterion(out, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            correct += (out.argmax(1) == y_batch).sum().item()
            total += X_batch.size(0)

        train_loss = total_loss / total
        train_acc  = correct / total

        # --- Evaluate ---
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.view(-1, 784)
                out = model(X_batch)
                loss = criterion(out, y_batch)
                total_loss += loss.item() * X_batch.size(0)
                correct += (out.argmax(1) == y_batch).sum().item()
                total += X_batch.size(0)

        test_loss = total_loss / total
        test_acc  = correct / total

        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['lr'].append(current_lr)

        if scheduler is not None:
            scheduler.step()

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"  Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | "
                  f"Test Acc: {test_acc:.4f} | LR: {current_lr:.6f}")

    return history

---

# Part 1: Vanilla SGD and the Optimization Landscape

**From the lecture (Slides 3–7, Eq. 8.1–8.3):**

Training a neural network means minimizing the empirical risk:

$$J(\theta) = \mathbb{E}_{(x,y) \sim \hat{p}_{\text{data}}} L(f(x; \theta), y)$$

**SGD** approximates the full gradient using a mini-batch $\mathbb{B}$:

$$\hat{g} = \frac{1}{|\mathbb{B}|} \sum_{i \in \mathbb{B}} \nabla_\theta L(f(x^{(i)}; \theta), y^{(i)})$$
$$\theta \leftarrow \theta - \epsilon \hat{g}$$

We start by implementing vanilla SGD from scratch on 2D loss surfaces so we can visualize the trajectory.

In [ ]:
# Define 2D loss surfaces for visualization

def quadratic_bowl(w):
    """Simple quadratic: f(x,y) = x^2 + y^2. Minimum at (0,0)."""
    return w[0]**2 + w[1]**2

def quadratic_bowl_grad(w):
    return np.array([2*w[0], 2*w[1]])


def ill_conditioned(w):
    """Ill-conditioned: f(x,y) = 50*x^2 + y^2. Narrow ravine along x-axis."""
    return 50*w[0]**2 + w[1]**2

def ill_conditioned_grad(w):
    return np.array([100*w[0], 2*w[1]])


def rosenbrock(w):
    """Rosenbrock: f(x,y) = (1-x)^2 + 100*(y-x^2)^2. Curved valley."""
    return (1 - w[0])**2 + 100*(w[1] - w[0]**2)**2

def rosenbrock_grad(w):
    dx = -2*(1 - w[0]) + 200*(w[1] - w[0]**2)*(-2*w[0])
    dy = 200*(w[1] - w[0]**2)
    return np.array([dx, dy])


def saddle_surface(w):
    """Saddle point: f(x,y) = x^2 - y^2. Saddle at (0,0)."""
    return w[0]**2 - w[1]**2

def saddle_surface_grad(w):
    return np.array([2*w[0], -2*w[1]])

In [ ]:
def sgd_vanilla(grad_fn, w_init, lr=0.01, steps=100):
    """Vanilla SGD. Returns trajectory of parameter values."""
    w = w_init.copy()
    trajectory = [w.copy()]
    for _ in range(steps):
        g = grad_fn(w)
        w = w - lr * g
        trajectory.append(w.copy())
    return np.array(trajectory)


# Run SGD on all four surfaces
surfaces = [
    ('Quadratic Bowl',   quadratic_bowl,   quadratic_bowl_grad,   np.array([4.0, 3.0]),  0.1,  50),
    ('Ill-Conditioned',  ill_conditioned,   ill_conditioned_grad,  np.array([4.0, 3.0]),  0.005, 100),
    ('Rosenbrock',       rosenbrock,        rosenbrock_grad,       np.array([-1.5, 1.5]), 0.001, 500),
    ('Saddle Point',     saddle_surface,    saddle_surface_grad,   np.array([0.5, 0.5]),  0.1,   30),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, (name, fn, grad_fn, w0, lr, steps) in zip(axes, surfaces):
    traj = sgd_vanilla(grad_fn, w0, lr=lr, steps=steps)

    # Create contour plot
    xr = [traj[:,0].min()-1, traj[:,0].max()+1]
    yr = [traj[:,1].min()-1, traj[:,1].max()+1]
    xx, yy = np.meshgrid(np.linspace(xr[0], xr[1], 200),
                         np.linspace(yr[0], yr[1], 200))
    zz = np.array([fn(np.array([x, y])) for x, y in zip(xx.ravel(), yy.ravel())]).reshape(xx.shape)

    ax.contour(xx, yy, zz, levels=30, cmap='viridis', alpha=0.6)
    ax.plot(traj[:,0], traj[:,1], 'r.-', markersize=3, linewidth=0.8, alpha=0.8)
    ax.plot(traj[0,0], traj[0,1], 'go', markersize=8, label='Start')
    ax.plot(traj[-1,0], traj[-1,1], 'r*', markersize=12, label='End')
    ax.set_title(f'{name}\nlr={lr}, {steps} steps')
    ax.legend(fontsize=8)
    ax.set_xlabel('w\u2081')
    ax.set_ylabel('w\u2082')

plt.suptitle('Vanilla SGD Trajectories on Different Loss Surfaces', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Quadratic bowl: smooth convergence to minimum")
print("- Ill-conditioned: oscillation along high-curvature direction (the ravine)")
print("- Rosenbrock: slow progress along curved valley")
print("- Saddle point: SGD escapes along the negative curvature direction")

---

# Part 2: Optimization Challenges

**From the lecture (Slides 8–12, Figures 8.1–8.4):**

**Ill-conditioning** (Eq. 8.8): Even with perfect gradient info, the gradient descent update can *increase* the cost when the Hessian has high condition number:

$$J(\theta - \epsilon g) \approx J(\theta) - \epsilon g^\top g + \frac{1}{2}\epsilon^2 g^\top H g$$

When $\frac{1}{2}\epsilon^2 g^\top H g > \epsilon g^\top g$, the second-order curvature term dominates and the loss goes **up**.

**Saddle points** are more common than local minima in high dimensions. For a function with $n$ parameters, a random critical point is exponentially more likely to be a saddle point than a local minimum.

**Vanishing/exploding gradients** occur when gradients are multiplied through many layers.

In [ ]:
# Demonstrate ill-conditioning: learning rate too high causes divergence

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

learning_rates = [0.001, 0.005, 0.015]
w0 = np.array([4.0, 3.0])

for ax, lr in zip(axes, learning_rates):
    traj = sgd_vanilla(ill_conditioned_grad, w0, lr=lr, steps=100)
    losses = [ill_conditioned(w) for w in traj]

    # Contour
    xr = max(abs(traj[:,0].min()), abs(traj[:,0].max())) + 1
    yr = max(abs(traj[:,1].min()), abs(traj[:,1].max())) + 1
    xr = min(xr, 10)  # clip for readability
    yr = min(yr, 10)
    xx, yy = np.meshgrid(np.linspace(-xr, xr, 200), np.linspace(-yr, yr, 200))
    zz = 50*xx**2 + yy**2

    ax.contour(xx, yy, zz, levels=30, cmap='viridis', alpha=0.5)
    # Clip trajectory to visible range
    vis = traj[(np.abs(traj[:,0]) < xr) & (np.abs(traj[:,1]) < yr)]
    ax.plot(vis[:,0], vis[:,1], 'r.-', markersize=3, linewidth=0.8)
    ax.plot(traj[0,0], traj[0,1], 'go', markersize=8)
    diverged = losses[-1] > losses[0]
    ax.set_title(f'lr={lr} | Final loss: {losses[-1]:.1f}\n{"DIVERGED!" if diverged else "Converging"}')
    ax.set_xlabel('w\u2081 (high curvature)')
    ax.set_ylabel('w\u2082 (low curvature)')
    ax.set_xlim(-xr, xr)
    ax.set_ylim(-yr, yr)

plt.suptitle('Ill-Conditioning: The Ravine Problem', fontsize=14)
plt.tight_layout()
plt.show()

print("The high curvature direction (w\u2081) needs a tiny learning rate,")
print("but the low curvature direction (w\u2082) needs a large one.")
print("\u2192 This is exactly why adaptive methods (AdaGrad, Adam) were invented.")

In [ ]:
# Visualize saddle points in 3D

fig = plt.figure(figsize=(14, 5))

# 3D saddle surface
ax1 = fig.add_subplot(121, projection='3d')
xx, yy = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(-2, 2, 100))
zz = xx**2 - yy**2
ax1.plot_surface(xx, yy, zz, cmap='coolwarm', alpha=0.7, edgecolor='none')
ax1.scatter([0], [0], [0], color='red', s=100, zorder=5)
ax1.set_title('Saddle Point: f(x,y) = x\u00b2 - y\u00b2')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('f(x,y)')

# Eigenvalue spectrum of Hessian at saddle points
ax2 = fig.add_subplot(122)
dims = np.arange(2, 102, 2)
# For a random symmetric matrix, fraction of negative eigenvalues ~ 0.5
# P(all positive) = P(local minimum) ~ (0.5)^n
p_local_min = 0.5 ** dims
p_saddle = 1 - p_local_min
ax2.semilogy(dims, p_local_min, 'b-', linewidth=2, label='P(local minimum)')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Number of dimensions')
ax2.set_ylabel('Probability')
ax2.set_title('Why Saddle Points Dominate in High Dimensions')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.annotate(f'n=100: P(local min) \u2248 {p_local_min[-1]:.1e}',
             xy=(100, p_local_min[-1]), xytext=(60, 1e-10),
             arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')

plt.tight_layout()
plt.show()

print("In n dimensions, a random critical point is a saddle with probability \u2248 1 - 0.5^n")
print("At n=100: P(saddle) \u2248 1.0, P(local minimum) \u2248 10\u207b\u00b3\u2070")

---

# Part 3: Momentum Methods From Scratch

**From the lecture (Slides 15–19, Eq. 8.15–8.21, Figure 8.5):**

**SGD with Momentum** (Algorithm 8.2): Introduces a velocity $v$ that accumulates past gradients:

$$v \leftarrow \alpha v - \epsilon \nabla_\theta J(\theta)$$
$$\theta \leftarrow \theta + v$$

The momentum coefficient $\alpha$ (typically 0.9) acts like a moving average. On a ravine, the oscillations cancel out while the consistent direction accumulates.

**Nesterov Momentum** (Algorithm 8.3): "Look ahead" — compute the gradient at the predicted future position:

$$v \leftarrow \alpha v - \epsilon \nabla_\theta J(\theta + \alpha v)$$
$$\theta \leftarrow \theta + v$$

This gives a corrective term that reduces overshooting.

In [ ]:
def sgd_momentum(grad_fn, w_init, lr=0.01, momentum=0.9, steps=100):
    """SGD with classical momentum (Algorithm 8.2)."""
    w = w_init.copy()
    v = np.zeros_like(w)
    trajectory = [w.copy()]
    for _ in range(steps):
        g = grad_fn(w)
        v = momentum * v - lr * g
        w = w + v
        trajectory.append(w.copy())
    return np.array(trajectory)


def sgd_nesterov(grad_fn, w_init, lr=0.01, momentum=0.9, steps=100):
    """SGD with Nesterov momentum (Algorithm 8.3)."""
    w = w_init.copy()
    v = np.zeros_like(w)
    trajectory = [w.copy()]
    for _ in range(steps):
        # Look ahead: compute gradient at the anticipated position
        g = grad_fn(w + momentum * v)
        v = momentum * v - lr * g
        w = w + v
        trajectory.append(w.copy())
    return np.array(trajectory)


# Compare all three on the ill-conditioned surface
w0 = np.array([4.0, 3.0])
lr, steps = 0.005, 80

traj_sgd      = sgd_vanilla(ill_conditioned_grad, w0, lr=lr, steps=steps)
traj_momentum = sgd_momentum(ill_conditioned_grad, w0, lr=lr, momentum=0.9, steps=steps)
traj_nesterov = sgd_nesterov(ill_conditioned_grad, w0, lr=lr, momentum=0.9, steps=steps)

print(f"Vanilla SGD   → final loss: {ill_conditioned(traj_sgd[-1]):.6f}")
print(f"Momentum      → final loss: {ill_conditioned(traj_momentum[-1]):.6f}")
print(f"Nesterov      → final loss: {ill_conditioned(traj_nesterov[-1]):.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
zz = 50*xx**2 + yy**2

methods = [
    ('Vanilla SGD',       traj_sgd,      'tomato'),
    ('Momentum (\u03b1=0.9)',  traj_momentum, 'royalblue'),
    ('Nesterov (\u03b1=0.9)',  traj_nesterov, 'forestgreen'),
]

for ax, (name, traj, color) in zip(axes, methods):
    ax.contour(xx, yy, zz, levels=30, cmap='viridis', alpha=0.4)
    ax.plot(traj[:,0], traj[:,1], '.-', color=color, markersize=3, linewidth=0.8, alpha=0.8)
    ax.plot(traj[0,0], traj[0,1], 'go', markersize=8)
    ax.plot(traj[-1,0], traj[-1,1], 'r*', markersize=12)
    ax.set_title(f'{name}\nFinal: ({traj[-1,0]:.3f}, {traj[-1,1]:.3f})')
    ax.set_xlabel('w\u2081')
    ax.set_ylabel('w\u2082')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-4, 4)

plt.suptitle('Momentum Tames the Ravine', fontsize=14)
plt.tight_layout()
plt.show()

# Loss curves comparison
fig, ax = plt.subplots(figsize=(10, 5))
for name, traj, color in methods:
    losses = [ill_conditioned(w) for w in traj]
    ax.semilogy(losses, color=color, label=name, linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Convergence Speed: Vanilla vs Momentum vs Nesterov')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# Part 4: Adaptive Learning Rates From Scratch

**From the lecture (Slides 20–27, Eq. 8.22–8.30):**

The key insight: **each parameter needs its own learning rate**.

**AdaGrad** (Algorithm 8.4) accumulates squared gradients:

$$r \leftarrow r + g \odot g, \quad \theta \leftarrow \theta - \frac{\epsilon}{\delta + \sqrt{r}} \odot g$$

Problem: $r$ grows monotonically → learning rate decays to zero.

**RMSProp** (Algorithm 8.5) fixes this with exponential moving average:

$$r \leftarrow \rho r + (1-\rho) g \odot g$$

**Adam** (Algorithm 8.7) combines momentum + RMSProp with bias correction:

$$s \leftarrow \rho_1 s + (1-\rho_1) g \quad \text{(1st moment)}$$
$$r \leftarrow \rho_2 r + (1-\rho_2) g \odot g \quad \text{(2nd moment)}$$
$$\hat{s} = \frac{s}{1-\rho_1^t}, \quad \hat{r} = \frac{r}{1-\rho_2^t} \quad \text{(bias correction)}$$
$$\theta \leftarrow \theta - \frac{\epsilon}{\sqrt{\hat{r}} + \delta} \hat{s}$$

In [ ]:
def adagrad(grad_fn, w_init, lr=0.5, steps=100, delta=1e-7):
    """AdaGrad optimizer (Algorithm 8.4)."""
    w = w_init.copy()
    r = np.zeros_like(w)  # accumulated squared gradients
    trajectory = [w.copy()]
    for _ in range(steps):
        g = grad_fn(w)
        r = r + g * g
        w = w - (lr / (delta + np.sqrt(r))) * g
        trajectory.append(w.copy())
    return np.array(trajectory)


def rmsprop(grad_fn, w_init, lr=0.01, rho=0.9, steps=100, delta=1e-6):
    """RMSProp optimizer (Algorithm 8.5)."""
    w = w_init.copy()
    r = np.zeros_like(w)
    trajectory = [w.copy()]
    for _ in range(steps):
        g = grad_fn(w)
        r = rho * r + (1 - rho) * g * g
        w = w - (lr / (delta + np.sqrt(r))) * g
        trajectory.append(w.copy())
    return np.array(trajectory)


def adam(grad_fn, w_init, lr=0.01, beta1=0.9, beta2=0.999, steps=100, delta=1e-8):
    """Adam optimizer (Algorithm 8.7)."""
    w = w_init.copy()
    s = np.zeros_like(w)  # 1st moment
    r = np.zeros_like(w)  # 2nd moment
    trajectory = [w.copy()]
    for t in range(1, steps + 1):
        g = grad_fn(w)
        s = beta1 * s + (1 - beta1) * g
        r = beta2 * r + (1 - beta2) * g * g
        s_hat = s / (1 - beta1**t)  # bias correction
        r_hat = r / (1 - beta2**t)
        w = w - (lr / (np.sqrt(r_hat) + delta)) * s_hat
        trajectory.append(w.copy())
    return np.array(trajectory)

In [ ]:
# Compare all optimizers on the ill-conditioned surface
w0 = np.array([4.0, 3.0])
steps = 100

trajectories = {
    'Vanilla SGD':  (sgd_vanilla(ill_conditioned_grad, w0, lr=0.005, steps=steps),  'tomato'),
    'Momentum':     (sgd_momentum(ill_conditioned_grad, w0, lr=0.005, momentum=0.9, steps=steps), 'orange'),
    'AdaGrad':      (adagrad(ill_conditioned_grad, w0, lr=0.5, steps=steps),        'royalblue'),
    'RMSProp':      (rmsprop(ill_conditioned_grad, w0, lr=0.05, steps=steps),       'forestgreen'),
    'Adam':         (adam(ill_conditioned_grad, w0, lr=0.1, steps=steps),            'purple'),
}

# Trajectory plot
fig, ax = plt.subplots(figsize=(10, 7))
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 4, 200))
zz = 50*xx**2 + yy**2
ax.contour(xx, yy, zz, levels=30, cmap='viridis', alpha=0.4)

for name, (traj, color) in trajectories.items():
    ax.plot(traj[:,0], traj[:,1], '.-', color=color, markersize=2, linewidth=1.2, label=name, alpha=0.8)

ax.plot(w0[0], w0[1], 'ko', markersize=10, label='Start')
ax.plot(0, 0, 'k*', markersize=15, label='Minimum')
ax.set_xlabel('w\u2081 (high curvature)')
ax.set_ylabel('w\u2082 (low curvature)')
ax.set_title('Optimizer Trajectories on Ill-Conditioned Surface\n(50x\u00b2 + y\u00b2)', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlim(-5, 5)
ax.set_ylim(-4, 4)
plt.tight_layout()
plt.show()

In [ ]:
# Convergence curves
fig, ax = plt.subplots(figsize=(10, 5))

for name, (traj, color) in trajectories.items():
    losses = [ill_conditioned(w) for w in traj]
    ax.semilogy(losses, color=color, label=f'{name} (final: {losses[-1]:.2e})', linewidth=2)

ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Convergence Speed: All Optimizers')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate AdaGrad's learning rate decay problem
w0 = np.array([4.0, 3.0])

traj_adagrad_short = adagrad(ill_conditioned_grad, w0, lr=0.5, steps=50)
traj_adagrad_long  = adagrad(ill_conditioned_grad, w0, lr=0.5, steps=500)
traj_rmsprop_long  = rmsprop(ill_conditioned_grad, w0, lr=0.05, steps=500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Effective learning rate over time
ax = axes[0]
r_adagrad = np.zeros(2)
r_rmsprop = np.zeros(2)
eff_lr_adagrad = []
eff_lr_rmsprop = []
w_ag, w_rms = w0.copy(), w0.copy()

for t in range(500):
    g_ag = ill_conditioned_grad(w_ag)
    r_adagrad += g_ag * g_ag
    eff_ag = 0.5 / (1e-7 + np.sqrt(r_adagrad))
    eff_lr_adagrad.append(eff_ag.mean())
    w_ag = w_ag - eff_ag * g_ag

    g_rms = ill_conditioned_grad(w_rms)
    r_rmsprop = 0.9 * r_rmsprop + 0.1 * g_rms * g_rms
    eff_rms = 0.05 / (1e-6 + np.sqrt(r_rmsprop))
    eff_lr_rmsprop.append(eff_rms.mean())
    w_rms = w_rms - eff_rms * g_rms

ax.semilogy(eff_lr_adagrad, color='royalblue', label='AdaGrad', linewidth=2)
ax.semilogy(eff_lr_rmsprop, color='forestgreen', label='RMSProp', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Effective LR (log scale)')
ax.set_title('AdaGrad LR Decays to Zero; RMSProp Stays Alive')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss comparison
ax = axes[1]
losses_ag  = [ill_conditioned(w) for w in traj_adagrad_long]
losses_rms = [ill_conditioned(w) for w in traj_rmsprop_long]
ax.semilogy(losses_ag, color='royalblue', label='AdaGrad', linewidth=2)
ax.semilogy(losses_rms, color='forestgreen', label='RMSProp', linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Long-Run: AdaGrad Stalls, RMSProp Converges')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 5: Parameter Initialization Strategies

**From the lecture (Slides 28–30):**

Initialization matters because the loss surface is non-convex — where you start determines where you end up.

**Xavier/Glorot initialization** (for tanh/sigmoid):
$$W \sim U\left[-\frac{\sqrt{6}}{\sqrt{n_{\text{in}} + n_{\text{out}}}}, \frac{\sqrt{6}}{\sqrt{n_{\text{in}} + n_{\text{out}}}}\right]$$

**He/Kaiming initialization** (for ReLU):
$$W \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}}}\right)$$

**Why it matters:** If weights are too small, signals vanish; too large, they explode. Proper initialization keeps the variance of activations stable across layers.

In [ ]:
# Visualize activation distributions through layers with different initializations

def forward_activations(model, x):
    """Return activations at each layer."""
    acts = []
    x = x.view(-1, 784)
    x = model.fc1(x)
    acts.append(('Layer 1 (pre-ReLU)', x.detach().numpy().ravel()))
    x = torch.relu(x)
    acts.append(('Layer 1 (post-ReLU)', x.detach().numpy().ravel()))
    x = model.fc2(x)
    acts.append(('Layer 2 (pre-ReLU)', x.detach().numpy().ravel()))
    x = torch.relu(x)
    acts.append(('Layer 2 (post-ReLU)', x.detach().numpy().ravel()))
    x = model.fc3(x)
    acts.append(('Output (logits)', x.detach().numpy().ravel()))
    return acts


# Get a batch of test data
sample_batch, _ = next(iter(test_loader))

init_methods = ['small_random', 'large_random', 'xavier', 'he']
init_labels  = ['Small Random (\u03c3=0.01)', 'Large Random (\u03c3=1.0)', 'Xavier/Glorot', 'He/Kaiming']

fig, axes = plt.subplots(len(init_methods), 5, figsize=(20, 12))

for row, (method, label) in enumerate(zip(init_methods, init_labels)):
    torch.manual_seed(42)
    model = MLP(init_method=method)
    model.eval()
    with torch.no_grad():
        acts = forward_activations(model, sample_batch)

    for col, (act_name, act_vals) in enumerate(acts):
        ax = axes[row, col]
        ax.hist(act_vals, bins=50, color='royalblue', alpha=0.7, density=True)
        ax.set_title(f'{act_name}\n\u03bc={act_vals.mean():.3f}, \u03c3={act_vals.std():.3f}', fontsize=9)
        if col == 0:
            ax.set_ylabel(label, fontsize=10, fontweight='bold')
        ax.set_xlim(-3, 3)

plt.suptitle('Activation Distributions Across Layers for Different Initializations', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Observations:")
print("- Small random: activations vanish (collapse to 0), network is effectively dead")
print("- Large random: activations saturate/explode, gradients blow up")
print("- Xavier: stable variance, designed for tanh/sigmoid")
print("- He/Kaiming: stable variance for ReLU networks (accounts for ReLU killing half the units)")

In [ ]:
# Train with different initializations and compare convergence
init_configs = [
    ('small_random', 'Small Random (\u03c3=0.01)', 'tomato'),
    ('large_random', 'Large Random (\u03c3=1.0)',  'gray'),
    ('xavier',       'Xavier/Glorot',          'royalblue'),
    ('he',           'He/Kaiming',             'forestgreen'),
]

init_histories = {}
for method, label, color in init_configs:
    torch.manual_seed(42)
    model = MLP(init_method=method)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    print(f"Training with {label}...")
    hist = train_model(model, train_loader, test_loader, optimizer, epochs=20, verbose=False)
    init_histories[label] = (hist, color)
    print(f"  Final test acc: {hist['test_acc'][-1]:.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, 21)

for label, (hist, color) in init_histories.items():
    axes[0].plot(ep, hist['train_loss'], color=color, label=label, linewidth=2)
    axes[1].plot(ep, [a * 100 for a in hist['test_acc']], color=color, label=label, linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss by Initialization')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy by Initialization')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 6: Learning Rate Scheduling

**From the lecture (Slides 31–32):**

A fixed learning rate is suboptimal: we want large steps early (explore) and small steps later (fine-tune near the minimum).

Common schedules:
- **Step decay**: $\epsilon_t = \epsilon_0 \cdot \gamma^{\lfloor t/s \rfloor}$ (e.g., halve every 10 epochs)
- **Cosine annealing**: $\epsilon_t = \epsilon_{\min} + \frac{1}{2}(\epsilon_0 - \epsilon_{\min})(1 + \cos(\frac{t}{T}\pi))$
- **Linear warmup**: ramp LR from 0 to $\epsilon_0$ over the first $W$ steps, then decay

**Warmup** is critical for Adam with large batch sizes: the second-moment estimate $r$ is unreliable in the first few steps (high bias), so a high LR can cause divergence.

In [ ]:
# Visualize different LR schedules

total_epochs = 50
lr_0 = 0.01

# Step decay
lr_step = [lr_0 * (0.5 ** (e // 15)) for e in range(total_epochs)]

# Cosine annealing
lr_min = 1e-5
lr_cosine = [lr_min + 0.5 * (lr_0 - lr_min) * (1 + np.cos(np.pi * e / total_epochs))
             for e in range(total_epochs)]

# Linear warmup + cosine decay
warmup_epochs = 5
lr_warmup_cosine = []
for e in range(total_epochs):
    if e < warmup_epochs:
        lr_warmup_cosine.append(lr_0 * (e + 1) / warmup_epochs)
    else:
        progress = (e - warmup_epochs) / (total_epochs - warmup_epochs)
        lr_warmup_cosine.append(lr_min + 0.5 * (lr_0 - lr_min) * (1 + np.cos(np.pi * progress)))

# Exponential decay
lr_exp = [lr_0 * (0.95 ** e) for e in range(total_epochs)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lr_step, label='Step Decay (\u00d70.5 every 15 epochs)', linewidth=2, color='tomato')
ax.plot(lr_cosine, label='Cosine Annealing', linewidth=2, color='royalblue')
ax.plot(lr_warmup_cosine, label='Warmup (5ep) + Cosine', linewidth=2, color='forestgreen')
ax.plot(lr_exp, label='Exponential Decay (\u03b3=0.95)', linewidth=2, color='purple')
ax.axhline(y=lr_0, color='gray', linestyle='--', alpha=0.5, label='Constant LR')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train with different LR schedules using PyTorch

schedule_configs = [
    ('Constant LR',           None,                                            'gray'),
    ('Step Decay',            lambda opt: optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5), 'tomato'),
    ('Cosine Annealing',      lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30, eta_min=1e-5), 'royalblue'),
    ('Exponential (\u03b3=0.95)', lambda opt: optim.lr_scheduler.ExponentialLR(opt, gamma=0.95), 'purple'),
]

sched_histories = {}
for name, sched_fn, color in schedule_configs:
    torch.manual_seed(42)
    model = MLP(init_method='he')
    optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    scheduler = sched_fn(optimizer) if sched_fn is not None else None
    print(f"Training with {name}...")
    hist = train_model(model, train_loader, test_loader, optimizer,
                       scheduler=scheduler, epochs=30, verbose=False)
    sched_histories[name] = (hist, color)
    print(f"  Final test acc: {hist['test_acc'][-1]:.4f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, 31)

for name, (hist, color) in sched_histories.items():
    axes[0].plot(ep, hist['train_loss'], color=color, label=name, linewidth=2)
    axes[1].plot(ep, [a * 100 for a in hist['test_acc']], color=color, label=name, linewidth=2)
    axes[2].plot(ep, hist['lr'], color=color, label=name, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('LR Over Time'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 7: Grand Optimizer Shootout on Fashion-MNIST

We now run 7 different optimizer configurations head-to-head on the same architecture (784 → 256 → 128 → 10), same initialization (He), and 30 epochs.

In [ ]:
optimizer_configs = [
    ('SGD (lr=0.01)',           lambda p: optim.SGD(p, lr=0.01),                        None, 'gray'),
    ('SGD + Momentum',          lambda p: optim.SGD(p, lr=0.01, momentum=0.9),          None, 'tomato'),
    ('SGD + Nesterov',          lambda p: optim.SGD(p, lr=0.01, momentum=0.9, nesterov=True), None, 'orange'),
    ('AdaGrad',                 lambda p: optim.Adagrad(p, lr=0.01),                    None, 'royalblue'),
    ('RMSProp',                 lambda p: optim.RMSprop(p, lr=0.001, alpha=0.9),        None, 'forestgreen'),
    ('Adam',                    lambda p: optim.Adam(p, lr=0.001),                      None, 'purple'),
    ('Adam + Cosine Schedule',  lambda p: optim.Adam(p, lr=0.001),
                                lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30, eta_min=1e-5), '#D95F02'),
]

shootout_results = {}

for name, opt_fn, sched_fn, color in optimizer_configs:
    torch.manual_seed(42)
    model = MLP(init_method='he')
    optimizer = opt_fn(model.parameters())
    scheduler = sched_fn(optimizer) if sched_fn is not None else None
    print(f"Training: {name}...")
    hist = train_model(model, train_loader, test_loader, optimizer,
                       scheduler=scheduler, epochs=30, verbose=False)
    shootout_results[name] = (hist, color)
    best_acc = max(hist['test_acc'])
    print(f"  Best test acc: {best_acc:.4f} (epoch {hist['test_acc'].index(best_acc)+1})")

print("\nDone! All 7 optimizers trained.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ep = range(1, 31)

for name, (hist, color) in shootout_results.items():
    axes[0].plot(ep, hist['train_loss'], color=color, label=name, linewidth=1.5, alpha=0.9)
    axes[1].plot(ep, [a * 100 for a in hist['test_acc']], color=color, label=name, linewidth=1.5, alpha=0.9)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Grand Optimizer Shootout on Fashion-MNIST', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary bar chart: best test accuracy per optimizer
names = []
best_accs = []
colors = []

for name, (hist, color) in shootout_results.items():
    names.append(name)
    best_accs.append(max(hist['test_acc']) * 100)
    colors.append(color)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, best_accs, color=colors, edgecolor='white', linewidth=0.5)

for bar, acc in zip(bars, best_accs):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{acc:.2f}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Best Test Accuracy (%)')
ax.set_title('Optimizer Comparison: Best Test Accuracy on Fashion-MNIST')
ax.set_xlim(left=min(best_accs) - 5)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---

# Part 8: Gradient Clipping

**From the lecture (Slides 32–33, Figure 8.3):**

When the loss surface has steep cliffs (common in RNNs), a single large gradient step can catapult parameters into a bad region. **Gradient clipping** caps the gradient norm:

$$\text{if } ||g|| > v: \quad g \leftarrow \frac{g \cdot v}{||g||}$$

This is a simple but essential technique for training deep/recurrent networks.

In [ ]:
def sgd_clipped(grad_fn, w_init, lr=0.01, clip_value=1.0, steps=100):
    """SGD with gradient clipping."""
    w = w_init.copy()
    trajectory = [w.copy()]
    for _ in range(steps):
        g = grad_fn(w)
        g_norm = np.linalg.norm(g)
        if g_norm > clip_value:
            g = g * clip_value / g_norm
        w = w - lr * g
        trajectory.append(w.copy())
    return np.array(trajectory)


# Cliff surface: steep in one region
def cliff_surface(w):
    return w[0]**2 + w[1]**2 + 50 * np.maximum(0, w[0] - 1)**2

def cliff_grad(w):
    dx = 2*w[0] + 100 * np.maximum(0, w[0] - 1)
    dy = 2*w[1]
    return np.array([dx, dy])

w0 = np.array([3.0, 2.0])

traj_noclip = sgd_vanilla(cliff_grad, w0, lr=0.01, steps=60)
traj_clip   = sgd_clipped(cliff_grad, w0, lr=0.01, clip_value=5.0, steps=60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xx, yy = np.meshgrid(np.linspace(-2, 4, 200), np.linspace(-3, 3, 200))
zz = xx**2 + yy**2 + 50 * np.maximum(0, xx - 1)**2

for ax, traj, name, color in [(axes[0], traj_noclip, 'No Clipping', 'tomato'),
                               (axes[1], traj_clip,   'Clipped (max_norm=5)', 'forestgreen')]:
    ax.contour(xx, yy, zz, levels=30, cmap='viridis', alpha=0.4)
    ax.plot(traj[:,0], traj[:,1], '.-', color=color, markersize=3, linewidth=1)
    ax.plot(traj[0,0], traj[0,1], 'go', markersize=8)
    ax.plot(traj[-1,0], traj[-1,1], 'r*', markersize=12)
    final_loss = cliff_surface(traj[-1])
    ax.set_title(f'{name}\nFinal loss: {final_loss:.4f}')
    ax.set_xlabel('w\u2081')
    ax.set_ylabel('w\u2082')

plt.suptitle('Gradient Clipping on a Cliff Surface', fontsize=14)
plt.tight_layout()
plt.show()

# Show gradient norms over steps
fig, ax = plt.subplots(figsize=(10, 4))
norms_noclip = [np.linalg.norm(cliff_grad(w)) for w in traj_noclip]
norms_clip   = [min(np.linalg.norm(cliff_grad(w)), 5.0) for w in traj_clip]
ax.plot(norms_noclip, color='tomato', label='No clipping', linewidth=2)
ax.plot(norms_clip, color='forestgreen', label='Clipped (max=5)', linewidth=2)
ax.axhline(y=5.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Step')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Norms With and Without Clipping')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# Part 9: Batch Normalization as an Optimization Aid

**From the lecture (Slides 33–37):**

Batch normalization normalizes each mini-batch of activations:

$$\hat{x}^{(k)} = \frac{x^{(k)} - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2 + \varepsilon}}$$

Then applies a learnable affine transform: $y^{(k)} = \gamma \hat{x}^{(k)} + \beta$.

**Why it helps optimization:**
- Smoother loss landscape → allows higher learning rates
- Reduces sensitivity to initialization
- Acts as a mild regularizer (noise from batch statistics)

In [ ]:
class MLPWithBN(nn.Module):
    """MLP with batch normalization."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 10)
        # He initialization
        for m in [self.fc1, self.fc2, self.fc3]:
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    def forward(self, x):
        x = x.view(-1, 784)
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        return self.fc3(x)


# Compare: with vs without BN, at different learning rates
learning_rates = [0.01, 0.05, 0.1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, lr in zip(axes, learning_rates):
    for use_bn, label, color in [(False, 'Without BN', 'tomato'), (True, 'With BN', 'royalblue')]:
        torch.manual_seed(42)
        model = MLPWithBN() if use_bn else MLP(init_method='he')
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
        hist = train_model(model, train_loader, test_loader, optimizer, epochs=20, verbose=False)
        ax.plot(range(1, 21), [a * 100 for a in hist['test_acc']],
                color=color, label=f'{label} ({max(hist["test_acc"])*100:.1f}%)', linewidth=2)

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'SGD+Momentum, lr={lr}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Batch Normalization Enables Higher Learning Rates', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key takeaway: BN allows much higher learning rates without divergence,")
print("leading to faster convergence and sometimes better final accuracy.")

---

# Summary

In this lab you implemented and compared the major optimization techniques from Chapter 8:

| Part | What You Did | Key Concept |
|------|-------------|-------------|
| 1 | Implemented vanilla SGD from scratch | Gradient descent on 2D loss surfaces |
| 2 | Explored optimization challenges | Ill-conditioning, saddle points, and why they matter |
| 3 | Built momentum and Nesterov from scratch | Velocity accumulation tames ravine oscillations |
| 4 | Implemented AdaGrad, RMSProp, Adam | Per-parameter adaptive learning rates |
| 5 | Compared initialization strategies | Xavier vs He vs random — proper init keeps signals alive |
| 6 | Experimented with LR schedules | Step decay, cosine annealing, warmup |
| 7 | Grand optimizer shootout | 7 configs on Fashion-MNIST, head-to-head |
| 8 | Gradient clipping | Essential for steep loss cliffs (RNNs) |
| 9 | Batch normalization | Smoother loss surface, higher LR tolerance |

**Practical rules of thumb:**
- **Default choice**: Adam with lr=0.001 — fast convergence, minimal tuning
- **If you have time to tune**: SGD + momentum + cosine schedule often generalizes better
- **Always use**: He initialization (for ReLU), gradient clipping (for deep/recurrent nets)
- **Batch normalization**: almost always helps, especially with higher LRs
- **LR schedule**: cosine annealing or step decay — never keep LR constant for long training

---

# Homework Exercises

### Exercise 1: Implement AdamW
**AdamW** (Loshchilov & Hutter, 2019) decouples weight decay from the adaptive gradient step. Implement it from scratch in NumPy:

$$\theta \leftarrow (1 - \lambda \epsilon) \theta - \frac{\epsilon}{\sqrt{\hat{r}} + \delta} \hat{s}$$

Compare AdamW vs Adam (with L2) on the ill-conditioned surface. Are the trajectories different? Why?

### Exercise 2: Momentum Coefficient Sweep
Run momentum SGD on the ill-conditioned surface with $\alpha \in \{0.0, 0.5, 0.9, 0.95, 0.99\}$. Plot:
1. Trajectories (contour plots)
2. Convergence curves (loss vs step)

At what value of $\alpha$ does the optimizer start overshooting? Why?

### Exercise 3: Learning Rate Warmup for Adam
Implement a linear warmup schedule for Adam (5 epochs warmup, then cosine decay). Train on Fashion-MNIST and compare:
- Adam with constant LR
- Adam with cosine decay only
- Adam with warmup + cosine decay

Plot training loss and test accuracy. Does warmup help? Under what batch size does it matter most? Try `batch_size \in \{64, 256, 1024\}`.

### Exercise 4: Second-Order Information (Bonus)
Compute and visualize the **Hessian eigenvalue spectrum** of a trained MLP at convergence. Use `torch.autograd.functional.hessian` on a small subset of parameters (e.g., first layer bias). What does the condition number tell you about optimization difficulty?

---

### Next Week

**Week 6: Convolutional Networks (Ch 9)**
- The convolution operation and its properties
- Pooling, padding, and stride
- Classic architectures: LeNet, AlexNet, VGG
- Building a CNN from scratch for image classification